# Evaluación de estructura Kimball — paso a paso

Notebook para revisar si la base de datos (SQL Server, Windows Authentication)
se acerca a una estructura dimensional (Ralph Kimball): tablas `DT_` (dimension)
y `FT_` (fact), relaciones inferidas por nombre de columna (no hay FKs
declaradas), y comparación entre los esquemas nuevos por dominio (`cli`,
`billetera`, `riesgos`, ...) contra el remanente de `dbo`.

Corre las celdas en orden. Cada sección explica qué mirar antes de pasar
a la siguiente.


In [ ]:
# Si te falta algo, descomenta e instala:
# %pip install sqlalchemy pyodbc pandas openpyxl networkx matplotlib


In [ ]:
from __future__ import annotations

import itertools
import re
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from sqlalchemy import create_engine, inspect
from sqlalchemy.engine import Engine

%matplotlib inline
pd.set_option("display.max_colwidth", None)


## 1. Conexión (Windows Authentication)

Con Windows Auth no se pasa usuario/contraseña: se usa
`Trusted_Connection=yes` en el connection string y el driver ODBC toma la
sesión de Windows actual. Ajusta `SERVER`, `DATABASE` y `DRIVER` según tu
entorno (`Get-OdbcDriver` en PowerShell o el Administrador de orígenes de
datos ODBC te dice qué driver tienes instalado).


In [ ]:
SERVER = "localhost"          # o "servidor\\instancia"
DATABASE = "TU_BASE"
DRIVER = "ODBC Driver 17 for SQL Server"

# Esquemas a evaluar. Incluye dbo explícitamente para compararlo con los
# esquemas nuevos por dominio.
SCHEMAS = ["dbo", "cli", "billetera", "riesgos"]

# Nombres de columnas demasiado genéricos para usarse en la inferencia de
# relaciones (generarían falsos positivos entre casi cualquier tabla).
COLUMNAS_GENERICAS = {
    "id", "created_at", "updated_at", "fecha_creacion", "fecha_actualizacion",
    "estado", "activo", "fecha", "nombre", "descripcion", "tipo",
}

conn_str = (
    f"mssql+pyodbc://@{SERVER}/{DATABASE}"
    f"?driver={DRIVER.replace(' ', '+')}&trusted_connection=yes"
)
engine = create_engine(conn_str)

# Prueba rápida de conexión
with engine.connect() as conn:
    version = conn.exec_driver_sql("SELECT @@VERSION").scalar()
print("Conectado OK.")
print(version)


## 2. Reflection del esquema

Extrae todas las tablas de los esquemas indicados, sus columnas, tipos y PK
declarada (si la hay). También detecta el prefijo `DT_`/`FT_` del nombre de
tabla, que vamos a usar como clasificación de verdad más adelante.

**Qué mirar:** que el número de tablas por esquema tenga sentido (si sale 0
en algún esquema, revisa el nombre exacto o permisos de `INFORMATION_SCHEMA`).


In [ ]:
PREFIJO_RE = re.compile(r"^(DT|FT)[_\-]?", re.IGNORECASE)


def detectar_tipo_por_prefijo(nombre_tabla: str) -> str | None:
    m = PREFIJO_RE.match(nombre_tabla)
    if not m:
        return None
    return "dimension" if m.group(1).upper() == "DT" else "fact"


@dataclass
class TablaInfo:
    schema: str
    nombre: str
    columnas: list[str] = field(default_factory=list)
    tipos: dict[str, str] = field(default_factory=dict)
    pk: list[str] = field(default_factory=list)
    n_filas: int | None = None
    tipo_declarado: str | None = None

    @property
    def nombre_completo(self) -> str:
        return f"{self.schema}.{self.nombre}"

    @property
    def ratio_numericas(self) -> float:
        if not self.columnas:
            return 0.0
        numericos = {"int", "bigint", "smallint", "tinyint", "decimal",
                     "numeric", "float", "real", "money", "smallmoney"}
        n_num = sum(1 for t in self.tipos.values()
                    if any(t.lower().startswith(n) for n in numericos))
        return n_num / len(self.columnas)


def reflejar_tablas(engine: Engine, schemas: list[str]) -> list[TablaInfo]:
    inspector = inspect(engine)
    tablas = []
    for schema in schemas:
        for nombre_tabla in inspector.get_table_names(schema=schema):
            info = TablaInfo(schema=schema, nombre=nombre_tabla)
            info.tipo_declarado = detectar_tipo_por_prefijo(nombre_tabla)
            columnas = inspector.get_columns(nombre_tabla, schema=schema)
            info.columnas = [c["name"] for c in columnas]
            info.tipos = {c["name"]: str(c["type"]) for c in columnas}
            try:
                pk = inspector.get_pk_constraint(nombre_tabla, schema=schema)
                info.pk = pk.get("constrained_columns", []) or []
            except Exception:
                info.pk = []
            tablas.append(info)
    return tablas


tablas = reflejar_tablas(engine, SCHEMAS)
print(f"Se encontraron {len(tablas)} tablas en {SCHEMAS}.")
pd.DataFrame([
    {"schema": t.schema, "tabla": t.nombre, "n_columnas": len(t.columnas),
     "tipo_por_prefijo": t.tipo_declarado}
    for t in tablas
])


In [ ]:
# Opcional: cuenta filas por tabla (puede ser lento en bases grandes).
# Corre esta celda solo si te sirve el dato; si no, sáltala.

def contar_filas(engine: Engine, tablas: list[TablaInfo]) -> None:
    with engine.connect() as conn:
        for t in tablas:
            try:
                result = conn.exec_driver_sql(
                    f"SELECT COUNT(*) FROM [{t.schema}].[{t.nombre}]"
                )
                t.n_filas = result.scalar()
            except Exception:
                t.n_filas = None

contar_filas(engine, tablas)
pd.DataFrame([{"tabla": t.nombre_completo, "n_filas": t.n_filas} for t in tablas]) \
    .sort_values("n_filas", ascending=False)


## 3. Inferencia de relaciones (no hay FKs declaradas)

Compara columnas entre todas las tablas y marca como relación cualquier
columna en común que no esté en `COLUMNAS_GENERICAS`. Esto hace de "FK
fantasma": es lo más parecido que tenemos a integridad referencial dado que
no está declarada en la base.

**Qué mirar:** si dos tablas que *deberían* relacionarse no aparecen aquí,
probablemente el nombre de columna difiere entre esquemas (ej.
`cliente_id` vs `id_cliente`) — vale la pena homogeneizar eso antes de
seguir.


In [ ]:
def inferir_relaciones(tablas: list[TablaInfo]) -> pd.DataFrame:
    filas = []
    for t1, t2 in itertools.combinations(tablas, 2):
        comunes = (set(t1.columnas) & set(t2.columnas)) - COLUMNAS_GENERICAS
        if comunes:
            filas.append({
                "tabla_a": t1.nombre_completo,
                "tabla_b": t2.nombre_completo,
                "columnas_comunes": ", ".join(sorted(comunes)),
                "n_columnas_comunes": len(comunes),
            })
    return pd.DataFrame(filas).sort_values(
        "n_columnas_comunes", ascending=False
    ) if filas else pd.DataFrame(columns=[
        "tabla_a", "tabla_b", "columnas_comunes", "n_columnas_comunes"
    ])


relaciones = inferir_relaciones(tablas)
print(f"{len(relaciones)} relaciones inferidas.")
relaciones.head(20)


## 4. Clasificación: prefijo `DT_`/`FT_` como fuente de verdad

El prefijo es una decisión de diseño explícita, así que se usa como
clasificación real. La heurística (ratio de columnas numéricas, cantidad de
columnas `*_id`) solo sirve para contrastar: si una tabla `FT_` no coincide
con la heurística, probablemente le faltan medidas reales o el grano no
está bien definido — vale la pena revisarla a mano.


In [ ]:
def clasificar_tablas(tablas: list[TablaInfo], relaciones: pd.DataFrame) -> pd.DataFrame:
    filas = []
    for t in tablas:
        n_relaciones = len(relaciones[
            (relaciones["tabla_a"] == t.nombre_completo)
            | (relaciones["tabla_b"] == t.nombre_completo)
        ])
        cols_id = [c for c in t.columnas if c.lower().endswith("id")]

        score_fact = t.ratio_numericas * 2 + min(len(cols_id), 5) * 0.5
        score_dim = (1 - t.ratio_numericas) * 2 + (1 if t.pk else 0)

        if score_fact > score_dim and t.ratio_numericas > 0.25:
            clasificacion_heuristica = "posible FACT"
        elif score_dim >= score_fact:
            clasificacion_heuristica = "posible DIMENSION"
        else:
            clasificacion_heuristica = "ambigua"

        heuristica_normalizada = clasificacion_heuristica.replace("posible ", "").lower()
        tipo_final = t.tipo_declarado or heuristica_normalizada
        coincide = (t.tipo_declarado == heuristica_normalizada) if t.tipo_declarado else None

        filas.append({
            "tabla": t.nombre_completo,
            "n_columnas": len(t.columnas),
            "ratio_numericas": round(t.ratio_numericas, 2),
            "n_cols_tipo_id": len(cols_id),
            "tiene_pk_declarada": bool(t.pk),
            "n_relaciones_inferidas": n_relaciones,
            "n_filas": t.n_filas,
            "tipo_por_prefijo": t.tipo_declarado,
            "clasificacion_heuristica": clasificacion_heuristica,
            "coincide_prefijo_con_heuristica": coincide,
            "tipo_final": tipo_final,
        })
    return pd.DataFrame(filas).sort_values("n_relaciones_inferidas", ascending=False)


clasificacion = clasificar_tablas(tablas, relaciones)
clasificacion


In [ ]:
# Tablas donde el prefijo NO coincide con la heurística — revisar a mano.
clasificacion[clasificacion["coincide_prefijo_con_heuristica"] == False]


## 5. Grafo real: ¿forma de estrella o red enredada?

Nodos = tablas (coloreadas por tipo declarado); aristas = relaciones
inferidas del paso 3. Si el diseño es un star schema limpio, cada `FT_`
debería conectar directo con varias `DT_` y no con otros `FT_`.

**Qué mirar en la imagen:** los cuadrados rojos (facts) deberían estar
rodeados de círculos azules (dimensions). Si ves cuadrados rojos conectados
entre sí, o mucho gris (tablas sin prefijo, probablemente `dbo`) mezclado
en medio, ahí está la evidencia de que la migración quedó a medias.


In [ ]:
def construir_grafo(tablas: list[TablaInfo], relaciones: pd.DataFrame) -> nx.Graph:
    grafo = nx.Graph()
    for t in tablas:
        grafo.add_node(t.nombre_completo, tipo=t.tipo_declarado or "sin_prefijo")
    for _, fila in relaciones.iterrows():
        grafo.add_edge(fila["tabla_a"], fila["tabla_b"],
                        columnas=fila["columnas_comunes"], peso=fila["n_columnas_comunes"])
    return grafo


def analizar_grafo_por_fact(grafo: nx.Graph) -> pd.DataFrame:
    filas = []
    for nodo, datos in grafo.nodes(data=True):
        if datos.get("tipo") != "fact":
            continue
        vecinos = list(grafo.neighbors(nodo))
        n_dim = sum(1 for v in vecinos if grafo.nodes[v].get("tipo") == "dimension")
        n_fact = sum(1 for v in vecinos if grafo.nodes[v].get("tipo") == "fact")
        n_otros = len(vecinos) - n_dim - n_fact
        filas.append({
            "fact": nodo,
            "n_dimensiones_conectadas": n_dim,
            "n_facts_conectados_directo": n_fact,
            "n_tablas_sin_prefijo_conectadas": n_otros,
            "grado_total": len(vecinos),
        })
    return pd.DataFrame(filas).sort_values("n_dimensiones_conectadas", ascending=False)


def detectar_dimensiones_huerfanas(grafo: nx.Graph) -> pd.DataFrame:
    filas = []
    for nodo, datos in grafo.nodes(data=True):
        if datos.get("tipo") != "dimension":
            continue
        vecinos = list(grafo.neighbors(nodo))
        conecta_fact_directo = any(grafo.nodes[v].get("tipo") == "fact" for v in vecinos)
        if not conecta_fact_directo:
            filas.append({
                "dimension": nodo,
                "grado_total": len(vecinos),
                "vecinos": ", ".join(vecinos) if vecinos else "(ninguno)",
            })
    return pd.DataFrame(filas)


grafo = construir_grafo(tablas, relaciones)
print(f"Grafo: {grafo.number_of_nodes()} nodos, {grafo.number_of_edges()} aristas.")


In [ ]:
color_por_tipo = {"fact": "#e74c3c", "dimension": "#3498db", "sin_prefijo": "#95a5a6"}
colores = [color_por_tipo.get(grafo.nodes[n].get("tipo"), "#95a5a6") for n in grafo.nodes]
formas_fact = [n for n in grafo.nodes if grafo.nodes[n].get("tipo") == "fact"]

plt.figure(figsize=(14, 10))
layout = nx.spring_layout(grafo, k=0.6, seed=42)
nx.draw_networkx_edges(grafo, layout, alpha=0.3)
nx.draw_networkx_nodes(grafo, layout, node_color=colores, node_size=500)
nx.draw_networkx_nodes(grafo, layout, nodelist=formas_fact, node_color="#e74c3c",
                        node_shape="s", node_size=600)
nx.draw_networkx_labels(grafo, layout, font_size=7)
plt.title("Grafo inferido: rojo=fact, azul=dimension, gris=sin prefijo")
plt.axis("off")
plt.tight_layout()
plt.savefig("grafo_kimball.png", dpi=150)
plt.show()


In [ ]:
analisis_facts = analizar_grafo_por_fact(grafo)
analisis_facts


In [ ]:
dims_huerfanas = detectar_dimensiones_huerfanas(grafo)
dims_huerfanas if not dims_huerfanas.empty else "Ninguna — todas las DT_ conectan directo con algún FT_."


## 6. Duplicidad entre esquemas (`dbo` vs. dominios nuevos)

Compara tablas de distinto esquema por similitud de nombre y solape de
columnas (índice de Jaccard). Es la señal de si la migración **conformó**
las dimensiones (una sola versión de `cliente`, por ejemplo) o si terminó
duplicando la misma entidad entre `dbo` y el esquema de dominio nuevo.


In [ ]:
def detectar_duplicidad_entre_esquemas(tablas: list[TablaInfo]) -> pd.DataFrame:
    filas = []
    for t1, t2 in itertools.combinations(tablas, 2):
        if t1.schema == t2.schema:
            continue
        set1, set2 = set(t1.columnas), set(t2.columnas)
        if not set1 or not set2:
            continue
        jaccard = len(set1 & set2) / len(set1 | set2)
        nombre_similar = (t1.nombre.lower() in t2.nombre.lower()
                           or t2.nombre.lower() in t1.nombre.lower())
        if jaccard > 0.4 or nombre_similar:
            filas.append({
                "tabla_a": t1.nombre_completo,
                "tabla_b": t2.nombre_completo,
                "similitud_columnas_jaccard": round(jaccard, 2),
                "nombre_similar": nombre_similar,
            })
    return pd.DataFrame(filas).sort_values(
        "similitud_columnas_jaccard", ascending=False
    ) if filas else pd.DataFrame(columns=[
        "tabla_a", "tabla_b", "similitud_columnas_jaccard", "nombre_similar"
    ])


duplicidad = detectar_duplicidad_entre_esquemas(tablas)
duplicidad


## 7. Fan-out de `dbo` (señal de esquema aún normalizado)

Las tablas normalizadas (estilo OLTP) suelen compartir columnas con muchas
otras tablas de forma "delgada" (una o dos columnas en común, pero muchas
contrapartes). Un fan-out alto en `dbo` es señal indirecta de que esas
tablas siguen en forma normalizada y no dimensional.


In [ ]:
def evaluar_dbo(relaciones: pd.DataFrame) -> pd.DataFrame:
    fan_out = {}
    for _, fila in relaciones.iterrows():
        for tabla in (fila["tabla_a"], fila["tabla_b"]):
            if tabla.startswith("dbo."):
                fan_out[tabla] = fan_out.get(tabla, 0) + 1
    return pd.DataFrame(
        [{"tabla_dbo": k, "n_tablas_relacionadas": v} for k, v in fan_out.items()]
    ).sort_values("n_tablas_relacionadas", ascending=False)


dbo_fanout = evaluar_dbo(relaciones)
dbo_fanout


## 8. Exportar reporte completo

Junta todo en un Excel con una hoja por análisis, para revisar con calma o
compartir con el equipo.


In [ ]:
with pd.ExcelWriter("evaluacion_kimball.xlsx") as writer:
    clasificacion.to_excel(writer, sheet_name="clasificacion", index=False)
    relaciones.to_excel(writer, sheet_name="relaciones_inferidas", index=False)
    analisis_facts.to_excel(writer, sheet_name="analisis_por_fact", index=False)
    dims_huerfanas.to_excel(writer, sheet_name="dimensiones_huerfanas", index=False)
    duplicidad.to_excel(writer, sheet_name="duplicidad_esquemas", index=False)
    dbo_fanout.to_excel(writer, sheet_name="fanout_dbo", index=False)

print("Reporte exportado a evaluacion_kimball.xlsx")
print("Grafo exportado a grafo_kimball.png")
